<!-- field-paper-role-banner -->
> 🟩 **정답·해설본** — 같은 셀 위치의 실행 가능한 참고 구현입니다. 복사하기 전에 수식과 tensor shape를 설명하세요.

# 그래프 학습 · 추천 시스템 · 04. GAT: 이웃마다 다른 주의 가중치

## Graph Attention Networks

- **원 논문:** [Graph Attention Networks](https://arxiv.org/abs/1710.10903)
- **저자 / 발표:** Petar Veličković, Guillem Cucurull, Arantxa Casanova, Adriana Romero, Pietro Liò, Yoshua Bengio · ICLR (2018)
- **난이도 / 예상 시간:** 중급 · 약 70분
- **선수 지식:** attention, masked softmax, GCN
- **로컬 학습 데이터:** `data/field_curriculum/graph_recommendation.npz`

이 자료는 핵심 메커니즘을 확인하는 **하드웨어 인식 교육용 미니 재현**입니다.
PyTorch 학습은 CUDA → Apple MPS → CPU 순으로 자동 선택하고, NumPy·환경 simulation 같은
CPU 연산은 그대로 유지합니다. 원 논문의 전체 데이터·모델·학습 예산·최종 benchmark를
재현했다는 뜻은 아닙니다.

## 재현 범위

**보존하는 핵심:** masked self-attention 계수와 single-head GAT layer를 구현해 이웃별 가중치를 관찰합니다.

**축소하거나 생략한 부분:** Cora/PPI multi-head 모형 대신 18노드 single-head 모형을 사용합니다.

완료 기준은 정답과 문자가 같은 코드가 아니라, TODO 구현·불변식 assertion·작은 metric 또는
시각화·마지막 해석을 모두 마치는 것입니다.

## 원 논문 ↔ 실습 지도

        절·수식·그림 번호는 위 primary source를 기준으로 합니다. 판본에 따라 페이지가 달라질 수
        있으므로 절 제목과 식 번호를 함께 확인하세요.

        | 원 논문의 위치 | 대응 코드·실험 | 확인할 증거 |
        |---|---|---|
        | §2, Eq. (1) | 공유 선형변환 Wh | 출력 차원 |
| §2, Eq. (2) | LeakyReLU attention score | edge별 e_ij |
| §2, Eq. (3) | neighborhood masked softmax | 비edge 0·행합 1 |
| §2, Eq. (4) | attention weighted aggregation | 분류 loss·attention heatmap |

### 구현 단계 1 · 실행 환경·데이터 계약 (setup 예외)

- **원문 위치:** 해당 없음 — import, seed, device, 로컬 데이터 로딩만 담당하며 논문의 학습 연산을 구현하지 않습니다.
- **Tensor shape 계약:** A [N,N], X [N,D] → attention [N,N] → hidden [N,H]
- **구현 이유:** 이후 셀의 비교가 재현 가능하도록 장치·dtype·seed와 입력 불변식을 먼저 고정합니다.
- **완료 증거:** 데이터 존재 여부, 입력 rank, 장치 선택 assertion과 환경 요약 출력이 통과해야 합니다.

In [ ]:
from pathlib import Path
import random

import matplotlib.pyplot as plt
import numpy as np
import torch
from torch import Tensor, nn
from torch.nn import functional as F
from llm_engineering_lab.acceleration import get_accelerator

random.seed(7)
np.random.seed(7)
torch.manual_seed(7)
ACCELERATOR = get_accelerator()
DEVICE = ACCELERATOR.device
if DEVICE.type == "cuda":
    torch.cuda.manual_seed_all(7)

DATA_PATH = Path("data/field_curriculum/graph_recommendation.npz")
assert DATA_PATH.exists(), f"먼저 로컬 학습 데이터를 준비하세요: {DATA_PATH}"
raw = np.load(DATA_PATH, allow_pickle=False)
adjacency = torch.tensor(raw["adjacency"], dtype=torch.float32)
features = torch.tensor(raw["features"], dtype=torch.float32)
labels = torch.tensor(raw["labels"], dtype=torch.long)
train_mask = torch.tensor(raw["train_mask"], dtype=torch.bool)
test_mask = torch.tensor(raw["test_mask"], dtype=torch.bool)
assert adjacency.ndim == 2 and adjacency.shape[0] == adjacency.shape[1] == len(features)
assert torch.allclose(adjacency, adjacency.T) and not torch.diagonal(adjacency).any()
edge_count = int(adjacency.sum().item() / 2)
print(f"nodes={len(features)}, edges={edge_count}, features={features.shape[1]}")
print(ACCELERATOR.summary())
print(
    "graph sampling/topology preprocessing stays on CPU, "
    "while neural training uses DEVICE"
)

## 포트폴리오 해설: 핵심 수식에서 실행 코드까지

$$\alpha_{ij}=\operatorname{softmax}_{j\in\mathcal N(i)}\!\left(
\operatorname{LeakyReLU}(a^\top[Wh_i\Vert Wh_j])\right)$$

- **기호 정의:** i는 수신 노드, j는 이웃, W는 투영, a는 attention 벡터입니다.
- **수식의 역할:** masked softmax로 이웃마다 다른 데이터 의존 메시지 가중치를 만듭니다.
- **구현 이유:** 원문 전체를 축약하더라도 위 관계가 결과를 만드는 핵심이므로,
  이 수식을 전처리·`forward`·loss 경계로 나누어 직접 구현했습니다.
- **Task/코드 대응:** Task P의 `preprocess_graph`, 논문 전용 클래스의
  `forward`, `compute_loss`, `training_step`이 §2, Eq. (1)을 구조화합니다.
- **입출력 shape:** 특징 [N, F]와 edge mask [N, N] → 표현 [N, H], attention [N, N]
- **평가:** attention 행합·비edge 0 여부, 분류 loss와 test accuracy
- **원문 대비 한계:** Cora/PPI multi-head 모형 대신 18노드 single-head 모형을 사용합니다.

코드를 읽을 때는 `Config → 전처리 → Module.forward → loss → train → evaluate`
순서로 추적하세요. 각 함수는 한 가지 책임만 갖도록 분리했습니다.

### 구현 단계 2 · 핵심 연산·모델

- **원문 위치:** §2, Eq. (3)
- **이 셀의 책임:** neighborhood masked softmax
- **Tensor shape 계약:** A [N,N], X [N,D] → attention [N,N] → hidden [N,H]
- **구현 이유:** 수식을 숨긴 고수준 호출 대신 핵심 변환과 shape 경계를 직접 추적합니다.
- **완료 증거:** 비edge 0·행합 1. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
def graph_attention(wh: Tensor, adj: Tensor, attn_vector: Tensor) -> Tensor:
    n = len(wh)
    left = wh[:, None, :].expand(n, n, -1)
    right = wh[None, :, :].expand(n, n, -1)
    score = F.leaky_relu(torch.cat([left, right], -1) @ attn_vector, negative_slope=0.2)
    mask = (adj + torch.eye(n, device=adj.device, dtype=adj.dtype)) > 0
    return torch.softmax(score.masked_fill(~mask, -1e9), dim=1)


probe_wh = features[:, :4]
probe_a = graph_attention(probe_wh, adjacency, torch.linspace(-0.2, 0.3, 8))
edge_mask = (adjacency + torch.eye(len(adjacency))) > 0
assert torch.allclose(probe_a.sum(1), torch.ones(len(adjacency)), atol=1e-5)
assert probe_a[~edge_mask].max() < 1e-7

### 구현 단계 3 · 핵심 연산·모델

- **원문 위치:** §2, Eq. (2)
- **이 셀의 책임:** LeakyReLU attention score
- **Tensor shape 계약:** A [N,N], X [N,D] → attention [N,N] → hidden [N,H]
- **구현 이유:** 수식을 숨긴 고수준 호출 대신 핵심 변환과 shape 경계를 직접 추적합니다.
- **완료 증거:** edge별 e_ij. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
class GATLayer(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.linear = nn.Linear(in_dim, out_dim, bias=False)
        self.attn = nn.Parameter(torch.randn(2 * out_dim) * 0.1)

    def forward(self, x, adj, return_attention=False):
        wh = self.linear(x)
        alpha = graph_attention(wh, adj, self.attn)
        out = alpha @ wh
        return (out, alpha) if return_attention else out


gat = GATLayer(features.shape[1], 12)
out, alpha = gat(features, adjacency, True)
assert out.shape == (len(features), 12)

### 구현 단계 4 · 평가·완료 증거

- **원문 위치:** §2, Eq. (4)
- **이 셀의 책임:** attention weighted aggregation
- **Tensor shape 계약:** A [N,N], X [N,D] → attention [N,N] → hidden [N,H]
- **구현 이유:** 학습 숫자 하나가 아니라 논문 주장에 대응하는 반증 가능 증거를 남깁니다.
- **완료 증거:** 분류 loss·attention heatmap. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
gat = ACCELERATOR.move(gat)
classifier = ACCELERATOR.move(nn.Linear(12, int(labels.max()) + 1))
features_device, adjacency_device, labels_device = ACCELERATOR.move(
    features, adjacency, labels
)
train_mask_device, test_mask_device = ACCELERATOR.move(train_mask, test_mask)
optimizer = torch.optim.Adam([*gat.parameters(), *classifier.parameters()], lr=0.035)
losses = []
for _ in range(160):
    logits = classifier(F.elu(gat(features_device, adjacency_device)))
    loss = F.cross_entropy(logits[train_mask_device], labels_device[train_mask_device])
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    losses.append(float(loss.detach().cpu()))
with torch.no_grad():
    h, alpha = gat(features_device, adjacency_device, True)
    pred = classifier(F.elu(h)).argmax(1)
    accuracy = float(
        (pred[test_mask_device] == labels_device[test_mask_device]).float().mean()
    )
assert losses[-1] < losses[0] and torch.allclose(
    alpha.sum(1), torch.ones(len(alpha), device=alpha.device), atol=1e-5
)
fig, ax = plt.subplots(1, 2, figsize=(8, 3))
ax[0].plot(losses)
ax[0].set_title("loss")
ax[1].imshow(alpha.detach().cpu(), cmap="magma")
ax[1].set_title("learned α_ij")
plt.tight_layout()
plt.show()
print("test accuracy:", round(accuracy, 3))

### Task P1 · 전처리와 핵심 모델

먼저 graph 입력의 shape와 정규화 규칙을 고정하고, 논문의 핵심 연산을
`nn.Module`의 `forward`와 `compute_loss`로 분리합니다.

### 구현 단계 5 · 핵심 연산·모델

- **원문 위치:** §2, Eq. (1) / §2, Eq. (2) / §2, Eq. (3) / §2, Eq. (4)
- **이 셀의 책임:** 공유 선형변환 Wh / LeakyReLU attention score / neighborhood masked softmax / attention weighted aggregation
- **Tensor shape 계약:** A [N,N], X [N,D] → attention [N,N] → hidden [N,H]
- **구현 이유:** 수식을 숨긴 고수준 호출 대신 핵심 변환과 shape 경계를 직접 추적합니다.
- **완료 증거:** 출력 차원 / edge별 e_ij / 비edge 0·행합 1 / 분류 loss·attention heatmap. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
from dataclasses import dataclass


@dataclass(frozen=True)
class PortfolioConfig:
    hidden_dim: int = 12
    learning_rate: float = 0.03
    steps: int = 18


def preprocess_graph(graph: Tensor) -> Tensor:
    with_self = graph + torch.eye(len(graph), dtype=graph.dtype)
    degree = with_self.sum(dim=1).clamp_min(1.0)
    inverse_root = degree.pow(-0.5)
    return inverse_root[:, None] * with_self * inverse_root[None, :]


class GatPortfolioModel(nn.Module):
    def __init__(self, config: PortfolioConfig):
        super().__init__()
        self.config = config
        self.input_projection = nn.Linear(features.shape[1], config.hidden_dim)
        self.attention = nn.Parameter(torch.zeros(config.hidden_dim * 2))

        self.hidden = nn.Linear(config.hidden_dim, config.hidden_dim)
        self.classifier = nn.Linear(config.hidden_dim, int(labels.max()) + 1)
        self.last_attention = None

    def forward(self, inputs: Tensor, graph: Tensor) -> Tensor:
        hidden = self.input_projection(inputs)

        left = hidden[:, None, :].expand(-1, len(hidden), -1)
        right = hidden[None, :, :].expand(len(hidden), -1, -1)
        pair = torch.cat([left, right], dim=-1)
        scores = F.leaky_relu(pair @ self.attention, negative_slope=0.2)
        mask = graph > 0
        scores = scores.masked_fill(~mask, -1e9)
        self.last_attention = torch.softmax(scores, dim=1)
        combined = self.last_attention @ hidden

        representation = F.relu(self.hidden(combined))
        return self.classifier(representation)

    def compute_loss(self, outputs: Tensor, targets: Tensor) -> Tensor:
        return F.cross_entropy(outputs[train_mask_device], targets[train_mask_device])

    def training_step(self, inputs: Tensor, graph: Tensor, targets: Tensor) -> Tensor:
        outputs = self(inputs, graph)
        return self.compute_loss(outputs, targets)

### Task P2 · objective와 update

`fit_portfolio_model`은 gradient가 흐르는 parameter와 한 step의 순서를
드러냅니다. 논문의 목적함수가 코드의 어느 tensor에 적용되는지 확인하세요.

### 구현 단계 6 · 목적함수·학습 update

- **원문 위치:** §2, Eq. (4)
- **이 셀의 책임:** attention weighted aggregation
- **Tensor shape 계약:** A [N,N], X [N,D] → attention [N,N] → hidden [N,H]
- **구현 이유:** forward 결과와 논문 objective, gradient update의 책임을 분리합니다.
- **완료 증거:** 분류 loss·attention heatmap. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
def fit_portfolio_model(
    model: GatPortfolioModel,
    inputs: Tensor,
    graph: Tensor,
    targets: Tensor,
) -> list[float]:
    optimizer = torch.optim.Adam(model.parameters(), lr=model.config.learning_rate)
    history = []
    for _ in range(model.config.steps):
        loss = model.training_step(inputs, graph, targets)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        history.append(float(loss.detach().cpu()))
    return history

### Task P3 · 평가와 한계 기록

loss와 별개의 논문 지표를 계산하고, 로컬 합성 graph에서 얻은 수치가
원 논문의 대규모 benchmark 결과를 대신하지 않는다는 점을 기록합니다.

### 구현 단계 7 · 평가·완료 증거

- **원문 위치:** §2, Eq. (4)
- **이 셀의 책임:** attention weighted aggregation
- **Tensor shape 계약:** A [N,N], X [N,D] → attention [N,N] → hidden [N,H]
- **구현 이유:** 학습 숫자 하나가 아니라 논문 주장에 대응하는 반증 가능 증거를 남깁니다.
- **완료 증거:** 분류 loss·attention heatmap. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
def evaluate_portfolio_model(
    model: GatPortfolioModel,
    inputs: Tensor,
    graph: Tensor,
    targets: Tensor,
) -> dict[str, float]:
    model.eval()
    with torch.no_grad():
        prediction = model(inputs, graph).argmax(dim=1)
        accuracy = (
            (prediction[test_mask_device] == targets[test_mask_device]).float().mean()
        )
    return {"test_accuracy": float(accuracy.cpu())}


features_device, labels_device = ACCELERATOR.move(features, labels)
train_mask_device, test_mask_device = ACCELERATOR.move(train_mask, test_mask)
portfolio_config = PortfolioConfig()
portfolio_graph = ACCELERATOR.move(preprocess_graph(adjacency))
portfolio_model = ACCELERATOR.move(GatPortfolioModel(portfolio_config))
portfolio_history = fit_portfolio_model(
    portfolio_model,
    features_device,
    portfolio_graph,
    labels_device,
)
portfolio_metrics = evaluate_portfolio_model(
    portfolio_model,
    features_device,
    portfolio_graph,
    labels_device,
)
assert np.isfinite(portfolio_history).all()
assert 0.0 <= portfolio_metrics["test_accuracy"] <= 1.0
print(portfolio_metrics)

## 정답 해석 가이드

핵심은 최종 숫자를 외우는 것이 아니라 **원문 위치 → 수식 → 코드 → 반증 가능한 증거**를
연결하는 것입니다. 이 노트북은 다음 원 규모 조건을 재현하지 않았습니다:

> Cora/PPI multi-head 모형 대신 18노드 single-head 모형을 사용합니다.

다음 실험에서는 데이터 크기, 모델 폭, 학습 step 중 하나만 바꾸고 동일 seed에서 비교하세요.